# Algoritmos de búsqueda en grafos

En este notebook trabajaremos sobre **un mismo problema** para comparar:

- Depth-First Search (DFS)
- Breadth-First Search (BFS)
- Uniform Cost Search (UCS)
- Greedy Best-First Search (GBF)
- A*

La idea central es observar que la estructura general de búsqueda cambia poco.  
Lo que cambia principalmente es **cómo se organiza la frontier**.

## 1. Problema de búsqueda

- Estado inicial: `A`
- Estado meta: `F`
- Los números en las aristas representan el costo de moverse entre estados.

![Grafo DFS](https://drive.google.com/uc?export=view&id=1Bu3wvTzHffMQVHmGfvekSdmtaatt12qJ)



## 2. Representación del problema

Como el grafo tiene costos, cada vecino se representa mediante una tupla:

```python
(estado_vecino, costo_de_la_arista)
```

El orden de los vecinos también importa para DFS y BFS.

In [1]:
graph = {
    "A": [("B", 2), ("C", 1)],
    "B": [("D", 2), ("E", 1)],
    "C": [("E", 3)],
    "D": [("H", 1)],
    "E": [("H", 2)],
    "H": []
}

start = "A"
goal = "H"

## 3. Estructuras comunes

Cada nodo de búsqueda guarda:

- `state`: estado actual.
- `parent`: nodo desde el cual se llegó.
- `cost`: costo acumulado \(g(n)\).

In [2]:
class Node:
    def __init__(self, state, parent=None, cost=0):
        self.state = state
        self.parent = parent
        self.cost = cost

    def __repr__(self):
        return f"{self.state}(g={self.cost})"

In [3]:
def reconstruct_path(node):
    path = []

    while node is not None:
        path.append(node.state)
        node = node.parent

    return list(reversed(path))

## 4. DFS

DFS utiliza una **pila LIFO**.

Los costos aparecen en el grafo, pero DFS no los utiliza para decidir qué nodo expandir.

In [4]:
class StackFrontier:
    def __init__(self):
        self.frontier = []

    def add(self, node):
        self.frontier.append(node)

    def contains_state(self, state):
        return any(node.state == state for node in self.frontier)

    def empty(self):
        return len(self.frontier) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        return self.frontier.pop()

    def states(self):
        return [node.state for node in self.frontier]

In [5]:
def depth_first_search(graph, start, goal, verbose=True):
    frontier = StackFrontier()
    frontier.add(Node(start))

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node = frontier.remove()
        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo: {node.state}")
            print(f"Frontier antes de agregar vecinos: {frontier.states()}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if (
                child_state not in explored
                and not frontier.contains_state(child_state)
            ):
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child)

        if verbose:
            print(f"Frontier después de expandir: {frontier.states()}")
            print("-" * 45)

    return None

In [6]:
dfs_result = depth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(dfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(dfs_result["path"]))
print("Costo del camino:", dfs_result["cost"])

Expandiendo: A
Frontier antes de agregar vecinos: []
Frontier después de expandir: ['B', 'C']
---------------------------------------------
Expandiendo: C
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'E']
---------------------------------------------
Expandiendo: E
Frontier antes de agregar vecinos: ['B']
Frontier después de expandir: ['B', 'H']
---------------------------------------------
Expandiendo: H
Frontier antes de agregar vecinos: ['B']
Orden de expansión: A → C → E → H
Camino encontrado: A → C → E → H
Costo del camino: 6


### Explicación — DFS

**Qué acaba de pasar:** la frontier es una **pila (LIFO)**. Al expandir `A` se agregaron `B` y `C` en ese orden, así que la pila quedó `[B, C]` y el siguiente en salir fue **C** (el último insertado), no B.

Traza:

| Paso | Frontier (tope a la derecha) | Se expande | Se agrega |
|---|---|---|---|
| 0 | `[A]` | — | — |
| 1 | `[B, C]` | A | B, C |
| 2 | `[B, E]` | C | E |
| 3 | `[B, H]` | E | H |
| 4 | `[B]` | H = meta ✓ | — |

**Resultado:** camino `A → C → E → H`, costo `1+3+2 = 6`.

**Punto clave para la exposición:** DFS **no es óptimo**. Existe `A → B → E → H` con costo 5, pero DFS nunca lo evaluó porque se hundió por la primera rama que tomó y devolvió la primera solución que encontró. Su ventaja es la memoria: solo guarda la rama actual, `O(bm)` frente a `O(b^d)` de BFS.


## 5. BFS — actividad

BFS utiliza una **cola FIFO**.

### Tareas

1. Complete el método `remove` de `QueueFrontier`.
2. Complete `breadth_first_search`.
3. Verifique el orden de expansión y el camino encontrado.
4. Compare el resultado con su solución manual.

In [7]:
class QueueFrontier(StackFrontier):
    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        return self.frontier.pop(0)


In [8]:
def breadth_first_search(graph, start, goal, verbose=True):
    frontier = QueueFrontier()
    frontier.add(Node(start))

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node = frontier.remove()
        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo: {node.state}")
            print(f"Frontier antes de agregar vecinos: {frontier.states()}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if (
                child_state not in explored
                and not frontier.contains_state(child_state)
            ):
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child)

        if verbose:
            print(f"Frontier después de expandir: {frontier.states()}")
            print("-" * 45)

    return None


In [9]:
# Ejecute esta celda cuando complete BFS.
bfs_result = breadth_first_search(graph, start, goal)

print("Orden de expansión:", " → ".join(bfs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(bfs_result["path"]))
print("Costo del camino:", bfs_result["cost"])

Expandiendo: A
Frontier antes de agregar vecinos: []
Frontier después de expandir: ['B', 'C']
---------------------------------------------
Expandiendo: B
Frontier antes de agregar vecinos: ['C']
Frontier después de expandir: ['C', 'D', 'E']
---------------------------------------------
Expandiendo: C
Frontier antes de agregar vecinos: ['D', 'E']
Frontier después de expandir: ['D', 'E']
---------------------------------------------
Expandiendo: D
Frontier antes de agregar vecinos: ['E']
Frontier después de expandir: ['E', 'H']
---------------------------------------------
Expandiendo: E
Frontier antes de agregar vecinos: ['H']
Frontier después de expandir: ['H']
---------------------------------------------
Expandiendo: H
Frontier antes de agregar vecinos: []
Orden de expansión: A → B → C → D → E → H
Camino encontrado: A → B → D → H
Costo del camino: 5


In [10]:
# Pruebas básicas para BFS
assert bfs_result["path"] == ["A", "B", "D", "H"]
assert bfs_result["expansion_order"] == ["A", "B", "C", "D", "E", "H"]

print("✓ BFS pasó las pruebas.")

✓ BFS pasó las pruebas.


### Explicación — BFS

**La única línea que cambia respecto a DFS** es `remove()`: en vez de `self.frontier.pop()` (último) usamos `self.frontier.pop(0)` (primero). Eso convierte la pila en una **cola FIFO** y el algoritmo pasa de hundirse a explorar **por niveles**.

Traza:

| Paso | Frontier (sale por la izquierda) | Se expande | Se agrega |
|---|---|---|---|
| 0 | `[A]` | — | — |
| 1 | `[B, C]` | A | B, C |
| 2 | `[C, D, E]` | B | D, E |
| 3 | `[D, E]` | C | — (E ya está en frontier) |
| 4 | `[E, H]` | D | H |
| 5 | `[H]` | E | — (H ya está) |
| 6 | `[]` | H = meta ✓ | — |

**Resultado:** camino `A → B → D → H`, costo 5, con 3 aristas.

**Punto clave:** BFS garantiza el camino con **menor número de aristas**, no el de menor costo. Aquí coincide que también cuesta 5, pero es coincidencia: si `B→D` costara 100, BFS igual devolvería este camino porque solo cuenta saltos.


## 6. UCS

UCS utiliza una **cola de prioridad** y siempre expande el nodo con menor costo acumulado:

$$
g(n)
$$

Si aparece un camino más barato hacia un estado que ya estaba en la frontier, se conserva el de menor costo.

### Frontier con prioridad

En **Uniform Cost Search (UCS)**, **Greedy Best-First Search (GBF)** y **A\*** la *frontier* ya no es una pila ni una cola, sino una **cola de prioridad**.

La prioridad determina cuál será el siguiente nodo en expandirse:

- **UCS:** prioridad = \(g(n)\)
- **GBF:** prioridad = \(h(n)\)
- **A\*:** prioridad = \(g(n)+h(n)\)

Esta implementación utiliza el módulo `heapq` de Python, que mantiene automáticamente el elemento con **menor prioridad** en la primera posición.

#### Componentes de la clase

- **`heap`**: almacena la cola de prioridad.
- **`counter`**: genera un número consecutivo para cada inserción. Se utiliza para desempatar cuando dos nodos tienen la misma prioridad, respetando el orden en que fueron agregados.
- **`add(node, priority)`**: inserta un nodo en la frontier con su prioridad correspondiente.
- **`remove()`**: extrae el nodo con la menor prioridad.
- **`empty()`**: indica si la frontier está vacía.

Cada elemento del heap se almacena como una tupla:

```python
(priority, insertion_order, node)
```

Por ejemplo,

```python
(5, 3, Node("E"))
```

significa:

- prioridad = **5**
- fue el **cuarto nodo** insertado (`3` porque el contador empieza en 0)
- corresponde al nodo **E**.

De esta forma, si dos nodos tienen la misma prioridad, se expandirá primero el que fue insertado antes.

### ¿Qué hace `heapq.heappush`?

La siguiente instrucción agrega un nuevo nodo a la **cola de prioridad** (`heap`):

```python
heapq.heappush(
    self.heap,
    (priority, next(self.counter), node)
)
```

Observa que **no se almacena únicamente el nodo**, sino una **tupla** con tres elementos:

```python
(priority, insertion_order, node)
```

donde:

- **`priority`**: valor utilizado para ordenar la frontier.
  - UCS: `g(n)`
  - GBF: `h(n)`
  - A*: `g(n) + h(n)`

- **`next(self.counter)`**: número consecutivo que indica el orden de inserción.
  Se utiliza para desempatar cuando dos nodos tienen la misma prioridad.

- **`node`**: objeto que contiene el estado, el padre y el costo acumulado.

---

### Ejemplo

Supongamos que insertamos los siguientes nodos:

```python
(5, 0, Node("B"))
(3, 1, Node("C"))
(5, 2, Node("D"))
```

La prioridad es el **primer elemento** de la tupla, por lo que el primer nodo en salir será:

```text
Node("C")
```

porque tiene prioridad **3**.

Posteriormente saldrán:

```text
Node("B")
Node("D")
```

Ambos tienen prioridad **5**, pero `B` fue insertado antes (`0 < 2`).

---

### ¿Por qué usar el contador?

Si almacenáramos únicamente

```python
(priority, node)
```

y dos nodos tuvieran la misma prioridad, Python intentaría comparar directamente los objetos `Node`, lo que produciría un error.

El contador evita este problema y garantiza un criterio de desempate consistente.

In [11]:
import heapq
from itertools import count

In [12]:
class PriorityFrontier:
    def __init__(self):
        self.heap = []
        self.counter = count()

    def add(self, node, priority):
        # El contador permite desempatar respetando el orden de inserción.
        heapq.heappush(
            self.heap,
            (priority, next(self.counter), node)
        )

    def empty(self):
        return len(self.heap) == 0

    def remove(self):
        if self.empty():
            raise Exception("La frontier está vacía.")

        priority, _, node = heapq.heappop(self.heap)
        return node, priority

In [13]:
def uniform_cost_search(graph, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=0)

    # Mejor costo conocido para cada estado.
    best_cost = {start: 0}

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost)

    return None

In [14]:
ucs_result = uniform_cost_search(graph, start, goal)

print("Orden de expansión:", " → ".join(ucs_result["expansion_order"]))
print("Camino encontrado:", " → ".join(ucs_result["path"]))
print("Costo óptimo:", ucs_result["cost"])

Expandiendo A: g=0
Expandiendo C: g=1
Expandiendo B: g=2
Expandiendo E: g=3
Expandiendo D: g=4
Expandiendo H: g=5
Orden de expansión: A → C → B → E → D → H
Camino encontrado: A → B → E → H
Costo óptimo: 5


### Explicación — UCS

UCS cambia la estructura de la frontier: ya no es pila ni cola, sino una **cola de prioridad** ordenada por `g(n)` (costo acumulado real desde el inicio).

Traza (nodo expandido y su `g`):

| Paso | Se expande | g(n) | Frontier resultante (estado: g) |
|---|---|---|---|
| 1 | A | 0 | B:2, C:1 |
| 2 | C | 1 | B:2, E:4 |
| 3 | B | 2 | E:3 (mejora sobre 4), D:4 |
| 4 | E | 3 | D:4, H:5 |
| 5 | D | 4 | H:5 |
| 6 | H | 5 | meta ✓ |

**Resultado:** camino `A → B → E → H`, costo **5 = óptimo**.

**Dos detalles de implementación que vale la pena señalar:**

1. `best_cost` guarda el mejor costo conocido de cada estado. Cuando por `B` llegamos a `E` con `g=3`, mejora el `g=4` que traía por `C`, así que se reinserta E con la ruta barata.
2. La línea `if node.cost != best_cost.get(node.state): continue` descarta las **entradas obsoletas** del heap (la vieja E con g=4). Como `heapq` no permite actualizar prioridades, la técnica estándar es insertar la versión nueva e ignorar la vieja al salir.

**Punto clave:** UCS **sí es óptimo** con costos no negativos. Es Dijkstra. Nótese que expandió C antes que B (g=1 < g=2) aunque C no sirve para el camino final: UCS expande en orden de costo, gastando trabajo en zonas baratas pero inútiles.


## 7. Heurística para GBF y A*

La heurística \(h(n)\) estima el costo restante desde cada nodo hasta la meta.

![Heurística](heuristica_busqueda.png)

In [15]:
heuristic = {
    "A": 4,
    "B": 3,
    "C": 2,
    "D": 2,
    "E": 1,
    "H": 0
}

## 8. Greedy Best-First Search — actividad

GBF utiliza únicamente:

$$
h(n)
$$

### Tareas

1. Reutilice `PriorityFrontier`.
2. Use la heurística como prioridad.
3. No use el costo acumulado para decidir qué nodo expandir.
4. Retorne el camino, orden de expansión y costo real del camino.

In [16]:
def greedy_best_first_search(graph, heuristic, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start), priority=heuristic[start])

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        if node.state in explored:
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: h={priority}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            if child_state not in explored:
                child = Node(
                    state=child_state,
                    parent=node,
                    cost=node.cost + edge_cost
                )
                frontier.add(child, priority=heuristic[child_state])

    return None


In [17]:
# Ejecute esta celda cuando complete GBF.
gbf_result = greedy_best_first_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(gbf_result["expansion_order"]))
print("Camino encontrado:", " → ".join(gbf_result["path"]))
print("Costo del camino:", gbf_result["cost"])

Expandiendo A: h=4
Expandiendo C: h=2
Expandiendo E: h=1
Expandiendo H: h=0
Orden de expansión: A → C → E → H
Camino encontrado: A → C → E → H
Costo del camino: 6


In [18]:
# Pruebas básicas para GBF
assert gbf_result["path"] == ["A", "C", "E", "H"]
assert gbf_result["expansion_order"] == ["A", "C", "E", "H"]
assert gbf_result["cost"] == 6

print("✓ GBF pasó las pruebas.")

✓ GBF pasó las pruebas.


### Explicación — Greedy Best-First

GBF también usa cola de prioridad, pero la prioridad es **únicamente `h(n)`**: el estimado de lo que falta hasta la meta. **Ignora por completo `g(n)`**, lo que ya se recorrió.

Traza:

| Paso | Se expande | h(n) | Frontier (estado: h) |
|---|---|---|---|
| 1 | A | 4 | B:3, C:2 |
| 2 | C | 2 | B:3, E:1 |
| 3 | E | 1 | B:3, H:0 |
| 4 | H | 0 | meta ✓ |

**Resultado:** camino `A → C → E → H`, costo real **6 — subóptimo**.

**Punto clave (esto es lo que hay que explicar bien):** en el paso 1, GBF eligió C porque `h(C)=2 < h(B)=3`. Pero la arista `A→C` cuesta 1 y `C→E` cuesta 3, mientras que `A→B→E` cuesta 2+1=3. GBF se dejó engañar por un nodo que *parecía* más cerca de la meta, sin mirar lo caro que resultaba llegar hasta él ni lo caro que era continuar.

**Ventaja:** es el más rápido — expandió solo 4 nodos, contra 6 de UCS. **Desventaja:** ni óptimo ni completo (en grafos con ciclos puede quedarse dando vueltas si no se lleva `explored`).


## 9. A* — actividad

A* combina costo acumulado y heurística:

$$
f(n)=g(n)+h(n)
$$

### Tareas

1. Adapte la implementación de UCS.
2. Mantenga `best_cost` para permitir mejoras de ruta.
3. Use `new_cost + heuristic[child_state]` como prioridad.
4. Verifique que encuentre un camino óptimo.

In [19]:
def a_star_search(graph, heuristic, start, goal, verbose=True):
    frontier = PriorityFrontier()
    frontier.add(Node(start, cost=0), priority=heuristic[start])

    best_cost = {start: 0}

    explored = set()
    expansion_order = []

    while not frontier.empty():
        node, priority = frontier.remove()

        # Ignorar entradas antiguas de la cola de prioridad.
        if node.cost != best_cost.get(node.state):
            continue

        expansion_order.append(node.state)

        if verbose:
            print(f"Expandiendo {node.state}: g={node.cost}, f={priority}")

        if node.state == goal:
            return {
                "path": reconstruct_path(node),
                "expansion_order": expansion_order,
                "cost": node.cost
            }

        explored.add(node.state)

        for child_state, edge_cost in graph[node.state]:
            new_cost = node.cost + edge_cost

            if new_cost < best_cost.get(child_state, float("inf")):
                best_cost[child_state] = new_cost

                child = Node(
                    state=child_state,
                    parent=node,
                    cost=new_cost
                )

                frontier.add(child, priority=new_cost + heuristic[child_state])

    return None


In [20]:
# Ejecute esta celda cuando complete A*.
astar_result = a_star_search(graph, heuristic, start, goal)

print("Orden de expansión:", " → ".join(astar_result["expansion_order"]))
print("Camino encontrado:", " → ".join(astar_result["path"]))
print("Costo óptimo:", astar_result["cost"])

Expandiendo A: g=0, f=4
Expandiendo C: g=1, f=3
Expandiendo B: g=2, f=5
Expandiendo E: g=3, f=4
Expandiendo H: g=5, f=5
Orden de expansión: A → C → B → E → H
Camino encontrado: A → B → E → H
Costo óptimo: 5


In [21]:
# Pruebas básicas para A*
assert astar_result["path"] in (
    ["A", "B", "D", "H"],
    ["A", "B", "E", "H"]
)
assert astar_result["cost"] == 5

print("✓ A* pasó las pruebas.")

✓ A* pasó las pruebas.


### Explicación — A*

A* es la síntesis de UCS y GBF: la prioridad es

$$f(n) = g(n) + h(n)$$

es decir, *lo que ya gasté* + *lo que estimo que falta*. Es literalmente el código de UCS con una sola línea cambiada: `priority = new_cost + heuristic[child_state]`.

Traza:

| Paso | Se expande | g | h | f = g+h | Frontier (estado: f) |
|---|---|---|---|---|---|
| 1 | A | 0 | 4 | 4 | B:5, C:5 |
| 2 | C | 1 | 2 | 3 | B:5, E:5 |
| 3 | B | 2 | 3 | 5 | E:4 (mejorado), D:6 |
| 4 | E | 3 | 1 | 4 | D:6, H:5 |
| 5 | H | 5 | 0 | 5 | meta ✓ |

**Resultado:** camino `A → B → E → H`, costo **5 = óptimo**, expandiendo 5 nodos (UCS necesitó 6).

**Punto clave:** A* combina lo mejor de ambos: es **óptimo** como UCS, pero **más dirigido** — se saltó la expansión de D porque su `f=6` ya excedía el `f=5` de la meta.

**Condiciones de optimalidad (esto es lo que preguntan en el parcial):**
- **Admisible:** `h(n) ≤ costo real de n a la meta`. Nunca sobrestima. Garantiza optimalidad en búsqueda en árbol.
- **Consistente:** `h(n) ≤ costo(n, n') + h(n')` para todo vecino n'. Garantiza optimalidad en búsqueda en grafo, y además que cada nodo se expanda una sola vez.

Verificación rápida en este grafo: h(A)=4 y el costo real óptimo de A a H es 5 → 4 ≤ 5 ✓. h(C)=2 y el real de C a H es 5 → ✓. Todas las h son admisibles, por eso A* encontró el óptimo.


## 10. Comparación final

| Algoritmo | Tipo de frontier | Prioridad | Camino | Costo | Orden de expansión |
|---|---|---|---|---:|---|
| DFS | Pila | LIFO | A → C → E → H | 6 | A, C, E, H |
| BFS | Cola | FIFO | A → B → D → H | 5 | A, B, C, D, E, H |
| UCS | Cola de prioridad | g(n) | A → B → E → H | 5 | A, C, B, E, D, H |
| GBF | Cola de prioridad | h(n) | A → C → E → H | 6 | A, C, E, H |
| A* | Cola de prioridad | g(n)+h(n) | A → B → E → H | 5 | A, C, B, E, H |

### Preguntas de cierre

1. **¿Qué algoritmos garantizan el camino de menor número de aristas?** BFS. Al expandir por niveles, el primer camino que encuentra al goal siempre tiene el menor número de saltos (no el menor costo).

2. **¿Qué algoritmos garantizan el camino de menor costo?** UCS y A* (si la heurística es admisible). DFS y BFS no lo garantizan porque ignoran los costos de las aristas; GBF tampoco porque solo mira h(n).

3. **¿Por qué GBF puede encontrar un camino subóptimo?** Porque decide qué expandir usando solo h(n), el estimado a la meta, sin considerar cuánto costó llegar hasta ahí (g(n)). Se deja engañar por un nodo que "parece" cercano a la meta aunque el camino real hasta él sea caro: aquí sigue A→C→E→H (costo 6) porque h(C)=2 < h(B)=3, sin ver que A→B→E→H cuesta solo 5.

4. **¿Qué ocurre con A* si h(n)=0 para todos los nodos?** A* se convierte en UCS: f(n)=g(n)+0=g(n), por lo que la prioridad pasa a depender únicamente del costo acumulado.

5. **¿Qué efecto tiene el orden de los vecinos en DFS y BFS?** Cambia el orden de expansión y, en DFS, puede cambiar completamente el camino encontrado (al ser el primero que llega a la meta, no el óptimo). En BFS afecta el orden de expansión pero no el número de aristas del camino óptimo, aunque sí puede afectar cuál camino de igual longitud se retorna si hay varios.


![Grafo DFS](https://drive.google.com/uc?export=view&id=1C-Gocif6ltAwtFuRoROrR7D5rfs7jjcc)

